# NB3 — Extração de Features (Segmento Completo 30min)

**Pipeline de Predição de Crises Epilépticas a partir de EEG**

## Fluxo

```
NB1 → data/signals/{dataset}__{pat}__ctx{N}.npz   (inter 30min + pre 30min, 256Hz)
NB2 → data/preictal_estimate.json                 (PRE_SEC — usado no NB4)
                         ↓
NB3 → para cada paciente:
        carrega todos os .npz de contexto
        mapeia canais → ordem padronizada (R5→R0)
        extrai 19 features/canal em janelas 30s/passo 15s
        salva X_pre, X_inter com timestamps t_pre, t_inter
        → data/features/{dataset}__{pat}.npz
                         ↓
NB4 → para cada janela W ∈ {W1=10, W2=15, W3=20, W4=25, W5=PRE_SEC individual}:
        filtra X_pre[t_pre >= -W] e X_inter[-W:]
        LOSO por contexto de crise
```

## Por que NB3 NÃO seleciona janelas pré-ictais

As janelas W são a **variável independente da Etapa 1 do NB4**. O correto é:
- **NB3** = preprocessing puro: features do segmento completo com timestamps `t_pre`
- **NB4** = experimento: filtra pelos últimos W segundos de `t_pre`

## Checkpoint por contexto

Se interrompido, o NB3 retoma de onde parou.
Delete `data/features/.cache` para reprocessar do zero.

## Ordenação de canais (R5→R0 via fatiamento de colunas)

| Colunas | N ch | Nível | Regiões |
|---------|------|-------|---------|
| 0–37    | 2    | R0    | Temporal (behind-ear) |
| 0–75    | 4    | R1    | Temporal completo |
| 0–208   | 11   | R2    | + Frontal |
| 0–246   | 13   | R3    | + Central |
| 0–284   | 15   | R4    | + Parietal |
| 0–322   | 17   | R5    | + Occipital (completo) |

SeizeIT2 (2 ch nativos): usa apenas R0.


In [1]:
import subprocess, sys
for pkg in ['tqdm', 'PyWavelets', 'scipy']:
    try: __import__(pkg.replace('PyWavelets','pywt'))
    except ImportError: subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
print('Dependências OK.')

Dependências OK.


## 1. Imports e configuração

In [2]:
import os, re, json, warnings
from pathlib import Path
from collections import defaultdict
import numpy as np
from scipy.stats import skew, kurtosis as kurt
from scipy.signal import welch
from scipy.integrate import trapezoid
import pywt

warnings.filterwarnings('ignore')
os.environ['LOKY_MAX_CPU_COUNT'] = '1'

try:
    from tqdm.auto import tqdm
    HAS_TQDM = True
except ImportError:
    HAS_TQDM = False
    def tqdm(it, **kw): return it

ROOT       = Path('data')
SIGNAL_DIR = ROOT / 'signals'
FEAT_DIR   = ROOT / 'features'
FEAT_DIR.mkdir(parents=True, exist_ok=True)
(ROOT / 'figures').mkdir(parents=True, exist_ok=True)

# Parâmetros de segmentação — FIXOS, idênticos ao NB2
SFREQ      = 256.0
WIN_SEG_S  = 30.0
STEP_SEG_S = 15.0
MAX_PRE_S  = 30 * 60
N_FEATS_CH = 19

# Slices de coluna por nível (NB4 usa para fatiar X)
LEVEL_SLICES = {
    'R0':  2 * N_FEATS_CH,   #  38 colunas
    'R1':  4 * N_FEATS_CH,   #  76 colunas
    'R2': 11 * N_FEATS_CH,   # 209 colunas
    'R3': 13 * N_FEATS_CH,   # 247 colunas
    'R4': 15 * N_FEATS_CH,   # 285 colunas
    'R5': 17 * N_FEATS_CH,   # 323 colunas
}

print('Configuração carregada.')
print(f'Janela de SEGMENTAÇÃO: {WIN_SEG_S}s / passo {STEP_SEG_S}s')
print(f'LEVEL_SLICES: {LEVEL_SLICES}')
print('NOTA: janelas W do experimento (10,15,20,25min + individual) são definidas NO NB4.')

Configuração carregada.
Janela de SEGMENTAÇÃO: 30.0s / passo 15.0s
LEVEL_SLICES: {'R0': 38, 'R1': 76, 'R2': 209, 'R3': 247, 'R4': 285, 'R5': 323}
NOTA: janelas W do experimento (10,15,20,25min + individual) são definidas NO NB4.


## 2. Ordenação de canais — R0 ⊂ R1 ⊂ … ⊂ R5

In [3]:
CH_STD = [
    'T7',  'T8',
    'P7',  'P8',
    'Fp1', 'Fp2', 'F3', 'F4', 'F7', 'F8', 'Fz',
    'C3',  'C4',
    'P3',  'P4',
    'O1',  'O2',
]

CH_CHBMIT = [
    'T7-P7',  'T8-P8',
    'P7-O1',  'P8-O2',
    'FP1-F7', 'FP2-F8', 'FP1-F3', 'FP2-F4', 'FZ-CZ', 'F7-T7', 'F8-T8',
    'F3-C3',  'F4-C4',
    'C3-P3',  'C4-P4',
    'P3-O1',  'P4-O2',
]

CH_ALIASES = {
    'T3':'T7','T4':'T8','T5':'P7','T6':'P8',
    'FP1':'Fp1','FP2':'Fp2','FZ':'Fz',
}

def normalize_ch(name):
    n = name.strip()
    for pfx in ['EEG ', 'EEG-', 'eeg_']:
        if n.upper().startswith(pfx.upper()): n = n[len(pfx):]
    for sfx in ['-REF','-A1','-A2','-LE','-A','-CLE','-LF','-AVG']:
        if n.upper().endswith(sfx.upper()): n = n[:-len(sfx)]
    n = n.strip()
    if n.upper() in CH_ALIASES: return CH_ALIASES[n.upper()]
    if '-' in n: return n.upper()
    return n[0].upper() + n[1:].lower() if len(n) > 1 else n.upper()

def map_channels(ch_names_avail, priority_list, max_ch=17):
    avail_norm = {normalize_ch(c): i for i, c in enumerate(ch_names_avail)}
    used_names, used_idx, n_miss = [], [], 0
    for orig in priority_list[:max_ch]:
        pn = normalize_ch(orig)
        if pn in avail_norm:
            used_names.append(ch_names_avail[avail_norm[pn]])
            used_idx.append(avail_norm[pn])
        else:
            used_names.append(f'[{orig}]')
            used_idx.append(None)
            n_miss += 1
    return used_names, used_idx, n_miss

def choose_priority(dataset):
    if dataset == 'CHBMIT':   return CH_CHBMIT
    if dataset == 'SeizeIT2': return None
    return CH_STD

print('Mapeamento de canais definido.')
print('Nota: canais ausentes são preenchidos com zeros (não removidos).')

Mapeamento de canais definido.
Nota: canais ausentes são preenchidos com zeros (não removidos).


## 3. Extração de features — 19 por canal

Temporal (7): std, var, rms, linha de comprimento, mobilidade, skewness, kurtosis
Espectral (7): delta, theta, alpha, beta, gamma, entropia espectral, beta relativo
Wavelet DWT db4 nível 4 (4): energia de cada subbanda
Complexidade Hjorth (1): complexidade

**Função idêntica ao NB2** — alterar em ambos simultaneamente.

In [4]:
def extract_features_window(window, sfreq):
    feats = []
    for ch in window:
        std = np.std(ch); diff1 = np.diff(ch); std_d1 = np.std(diff1)
        mob = std_d1 / (std + 1e-10)
        feats += [std, np.var(ch), float(np.sqrt(np.mean(ch**2))),
                  float(np.sum(np.abs(diff1))), float(mob),
                  float(skew(ch)), float(kurt(ch))]
        f, psd = welch(ch, fs=sfreq, nperseg=min(len(ch), int(sfreq*4)))
        def bp(lo, hi):
            m = (f>=lo)&(f<=hi)
            return float(trapezoid(psd[m], f[m])) if m.any() else 0.0
        d,t,a,b,g = bp(.5,4),bp(4,8),bp(8,13),bp(13,30),bp(30,40)
        tot = d+t+a+b+g+1e-10
        pn  = psd/(psd.sum()+1e-10); pn = pn[pn>0]
        feats += [d,t,a,b,g, float(-np.sum(pn*np.log2(pn))), float(b/tot)]
        c = pywt.wavedec(ch, 'db4', level=4)
        feats += [float(np.sum(c[4]**2)), float(np.sum(c[3]**2)),
                  float(np.sum(c[2]**2)), float(np.sum(c[1]**2))]
        diff2 = np.diff(diff1); mob_d1 = np.std(diff2)/(std_d1+1e-10)
        feats.append(float(mob_d1/(mob+1e-10)))
    return np.array(feats, dtype=np.float32)

def segment_signal(signal, sfreq, win_s=WIN_SEG_S, step_s=STEP_SEG_S):
    """Segmenta (n_ch, n_samp) em janelas e extrai features.
    Retorna feat_matrix (n_win, n_ch*19) e t_centers_s (tempo bruto desde o início).
    Convertido para referencial negativo em process_patient.
    """
    win_n  = int(win_s  * sfreq)
    step_n = int(step_s * sfreq)
    n_samp = signal.shape[1]
    feats, centers = [], []
    for start in range(0, n_samp - win_n + 1, step_n):
        feats.append(extract_features_window(signal[:, start:start+win_n], sfreq))
        centers.append((start + win_n/2) / sfreq)
    n_ch = signal.shape[0]
    if feats:
        return np.array(feats, dtype=np.float32), np.array(centers)
    return np.empty((0, n_ch*N_FEATS_CH), dtype=np.float32), np.array([])

print(f'Features OK — {N_FEATS_CH} por canal, janela {WIN_SEG_S}s.')

Features OK — 19 por canal, janela 30.0s.


## 4. Processamento por paciente

In [5]:
def process_patient(dataset, pat, npz_files, overwrite=False, progress_bar=None):
    out_path  = FEAT_DIR / f'{dataset}__{pat}.npz'
    cache_dir = FEAT_DIR / '.cache' / f'{dataset}__{pat}'
    if out_path.exists() and not overwrite:
        if progress_bar: progress_bar.update(len(npz_files))
        return str(out_path), 'ja existe'
    cache_dir.mkdir(parents=True, exist_ok=True)
    priority   = choose_priority(dataset)
    is_seizeit = (dataset == 'SeizeIT2')
    ch_order   = None
    n_miss_total = 0

    for npz_path in sorted(npz_files):
        m = re.search(r'ctx(\d+)', npz_path.stem)
        if not m:
            if progress_bar: progress_bar.update(1)
            continue
        ctx_id     = int(m.group(1))
        cache_path = cache_dir / f'ctx{ctx_id:03d}.npz'
        if cache_path.exists() and not overwrite:
            if ch_order is None:
                try:
                    tmp = np.load(cache_path, allow_pickle=True)
                    ch_order = list(tmp['ch_order'])
                except: pass
            if progress_bar:
                progress_bar.set_postfix_str(f'ctx{ctx_id:03d} cached')
                progress_bar.update(1)
            continue
        try:
            data = np.load(npz_path, allow_pickle=True)
        except Exception as e:
            tqdm.write(f'    Erro {npz_path.name}: {e}')
            if progress_bar: progress_bar.update(1)
            continue
        ch_orig   = list(data['ch_names'])
        sfreq     = float(data['sfreq'])
        inter_raw = data['inter'].astype(np.float32)
        pre_raw   = data['pre'].astype(np.float32)
        if is_seizeit:
            n_use   = min(2, inter_raw.shape[0])
            inter_m = inter_raw[:n_use]
            pre_m   = pre_raw[:n_use]
            if ch_order is None: ch_order = ch_orig[:n_use]
        else:
            names_o, idx_o, n_miss = map_channels(ch_orig, priority)
            n_miss_total += n_miss
            n_out   = len(names_o)
            inter_m = np.zeros((n_out, inter_raw.shape[1]), dtype=np.float32)
            pre_m   = np.zeros((n_out, pre_raw.shape[1]),   dtype=np.float32)
            for j, idx in enumerate(idx_o):
                if idx is not None and idx < inter_raw.shape[0]:
                    inter_m[j] = inter_raw[idx]
                    pre_m[j]   = pre_raw[idx]
            if ch_order is None: ch_order = names_o
        if inter_m.shape[1] < int(WIN_SEG_S * sfreq):
            tqdm.write(f'    ctx{ctx_id:03d}: sinal curto demais, ignorado')
            if progress_bar: progress_bar.update(1)
            continue
        Xi, ti = segment_signal(inter_m, sfreq)
        Xp, tp = segment_signal(pre_m,   sfreq)
        if len(Xi) == 0 or len(Xp) == 0:
            if progress_bar: progress_bar.update(1)
            continue
        pre_dur_s     = pre_m.shape[1] / sfreq
        tp_from_onset = tp - pre_dur_s       # negativo: segundos antes do onset
        inter_dur_s   = inter_m.shape[1] / sfreq
        ti_from_end   = ti - inter_dur_s     # negativo: segundos antes do fim
        np.savez_compressed(cache_path,
            Xi=Xi, Xp=Xp, ti=ti_from_end, tp=tp_from_onset,
            ctx_id=np.array(ctx_id), ch_order=np.array(ch_order))
        if progress_bar:
            progress_bar.set_postfix_str(f'ctx{ctx_id:03d} OK')
            progress_bar.update(1)

    cache_files = sorted(cache_dir.glob('ctx*.npz'))
    if not cache_files: return None, 'sem contextos processáveis'

    all_Xi, all_Xp, all_ti, all_tp, all_cti, all_ctp = [], [], [], [], [], []
    for cf in cache_files:
        try:
            with np.load(cf, allow_pickle=True) as c:
                ctx_id = int(c['ctx_id'])
                xi = c['Xi'].copy(); ti = c['ti'].copy()
                xp = c['Xp'].copy(); tp = c['tp'].copy()
                ch_o = list(c['ch_order'])
            all_Xi.append(xi); all_ti.append(ti)
            all_Xp.append(xp); all_tp.append(tp)
            all_cti.append(np.full(len(xi), ctx_id, dtype=np.int32))
            all_ctp.append(np.full(len(xp), ctx_id, dtype=np.int32))
            if ch_order is None: ch_order = ch_o
        except Exception as e:
            tqdm.write(f'    Erro ao ler cache {cf.name}: {e}')

    if not all_Xi: return None, 'cache vazio após merge'

    X_inter       = np.vstack(all_Xi)
    X_pre         = np.vstack(all_Xp)
    t_inter       = np.concatenate(all_ti)
    t_pre         = np.concatenate(all_tp)
    ctx_ids_inter = np.concatenate(all_cti)
    ctx_ids_pre   = np.concatenate(all_ctp)

    meta = json.dumps({'dataset': dataset, 'paciente': pat,
                       'ch_order': ch_order, 'level_slices': LEVEL_SLICES,
                       'n_miss_total': n_miss_total, 'sfreq': SFREQ,
                       'win_seg_s': WIN_SEG_S, 'step_seg_s': STEP_SEG_S})

    np.savez_compressed(out_path,
        X_pre=X_pre, X_inter=X_inter,
        t_pre=t_pre, t_inter=t_inter,
        ctx_ids_pre=ctx_ids_pre, ctx_ids_inter=ctx_ids_inter,
        meta=np.array(meta))
    return str(out_path), f'{X_pre.shape[0]} win pré / {X_inter.shape[0]} win inter'

print('process_patient definido.')

process_patient definido.


## 5. Execução — todos os pacientes

In [6]:
all_npz = defaultdict(list)
for f in sorted(SIGNAL_DIR.glob('*.npz')):
    m = re.match(r'(.+?)__(.+?)__ctx', f.stem)
    if m: all_npz[(m.group(1), m.group(2))].append(f)

print(f'Pacientes encontrados: {len(all_npz)}')
for (ds, pat), files in sorted(all_npz.items()):
    print(f'  {ds:<10} {pat:<12} {len(files)} contextos')

print()
results_nb3 = {}
with tqdm(total=sum(len(v) for v in all_npz.values()), desc='NB3 — features') as pbar:
    for (ds, pat), files in sorted(all_npz.items()):
        out, status = process_patient(ds, pat, files, overwrite=False, progress_bar=pbar)
        results_nb3[(ds, pat)] = status

print()
print('RESULTADO POR PACIENTE')
print('='*60)
for (ds, pat), status in sorted(results_nb3.items()):
    print(f'  {ds:<10} {pat:<12} -> {status}')

Pacientes encontrados: 26
  CHBMIT     chb03        3 contextos
  CHBMIT     chb05        3 contextos
  CHBMIT     chb06        7 contextos
  CHBMIT     chb07        3 contextos
  CHBMIT     chb08        5 contextos
  CHBMIT     chb10        6 contextos
  CHBMIT     chb14        5 contextos
  Mendeley   p10          2 contextos
  Mendeley   p11          2 contextos
  Mendeley   p12          2 contextos
  Mendeley   p13          2 contextos
  Mendeley   p15          3 contextos
  SeizeIT2   sub-002      5 contextos
  SeizeIT2   sub-034      5 contextos
  SeizeIT2   sub-035      8 contextos
  SeizeIT2   sub-039      13 contextos
  SeizeIT2   sub-047      7 contextos
  SeizeIT2   sub-073      16 contextos
  SeizeIT2   sub-103      7 contextos
  Siena      PN01         2 contextos
  Siena      PN05         2 contextos
  Siena      PN06         4 contextos
  Siena      PN09         3 contextos
  Siena      PN10         5 contextos
  Siena      PN13         3 contextos
  Siena      PN14     

NB3 — features: 100%|██████████| 127/127 [21:17<00:00, 10.06s/it, ctx004 OK]


RESULTADO POR PACIENTE
  CHBMIT     chb03        -> 357 win pré / 717 win inter
  CHBMIT     chb05        -> 357 win pré / 717 win inter
  CHBMIT     chb06        -> 833 win pré / 4193 win inter
  CHBMIT     chb07        -> 357 win pré / 1797 win inter
  CHBMIT     chb08        -> 595 win pré / 1195 win inter
  CHBMIT     chb10        -> 714 win pré / 2875 win inter
  CHBMIT     chb14        -> 595 win pré / 1195 win inter
  Mendeley   p10          -> 238 win pré / 477 win inter
  Mendeley   p11          -> 238 win pré / 420 win inter
  Mendeley   p12          -> 238 win pré / 381 win inter
  Mendeley   p13          -> 238 win pré / 479 win inter
  Mendeley   p15          -> 357 win pré / 705 win inter
  SeizeIT2   sub-002      -> 595 win pré / 2595 win inter
  SeizeIT2   sub-034      -> 595 win pré / 2127 win inter
  SeizeIT2   sub-035      -> 952 win pré / 4267 win inter
  SeizeIT2   sub-039      -> 1547 win pré / 5388 win inter
  SeizeIT2   sub-047      -> 833 win pré / 4193 win in

## 6. Verificação de sanidade

In [7]:
print('VERIFICAÇÃO DE SANIDADE')
print('='*70)
for fp in sorted(FEAT_DIR.glob('*.npz')):
    try:
        d    = np.load(fp, allow_pickle=True)
        meta = json.loads(str(d['meta']))
        n_pre   = d['X_pre'].shape[0]
        n_inter = d['X_inter'].shape[0]
        n_cols  = d['X_pre'].shape[1]
        n_ctx_p = len(np.unique(d['ctx_ids_pre']))
        n_ctx_i = len(np.unique(d['ctx_ids_inter']))
        ok = 'OK' if n_ctx_p == n_ctx_i else 'AVISO ctx mismatch'
        print(f'  {ok} {meta["dataset"]:<10} {meta["paciente"]:<12} '
              f'pre={n_pre}win/{n_ctx_p}ctx  inter={n_inter}win/{n_ctx_i}ctx  cols={n_cols}')
    except Exception as e:
        print(f'  ERRO {fp.name}: {e}')

VERIFICAÇÃO DE SANIDADE
  OK CHBMIT     chb03        pre=357win/3ctx  inter=717win/3ctx  cols=323
  OK CHBMIT     chb05        pre=357win/3ctx  inter=717win/3ctx  cols=323
  OK CHBMIT     chb06        pre=833win/7ctx  inter=4193win/7ctx  cols=323
  OK CHBMIT     chb07        pre=357win/3ctx  inter=1797win/3ctx  cols=323
  OK CHBMIT     chb08        pre=595win/5ctx  inter=1195win/5ctx  cols=323
  OK CHBMIT     chb10        pre=714win/6ctx  inter=2875win/6ctx  cols=323
  OK CHBMIT     chb14        pre=595win/5ctx  inter=1195win/5ctx  cols=323
  OK Mendeley   p10          pre=238win/2ctx  inter=477win/2ctx  cols=323
  OK Mendeley   p11          pre=238win/2ctx  inter=420win/2ctx  cols=323
  OK Mendeley   p12          pre=238win/2ctx  inter=381win/2ctx  cols=323
  OK Mendeley   p13          pre=238win/2ctx  inter=479win/2ctx  cols=323
  OK Mendeley   p15          pre=357win/3ctx  inter=705win/3ctx  cols=323
  OK SeizeIT2   sub-002      pre=595win/5ctx  inter=2595win/5ctx  cols=38
  OK Seiz